# 07 · Real-World Async I/O: Sockets, Streams (TCP Servers & Clients), Subprocesses, HTTP & Files

**Module 13 – Concurrency & asyncio** · Estimated time: 60 min

### 🎯 What you will learn
- Network basics: TCP vs UDP, ports, the `socket` module (blocking)
- **asyncio streams**: build a concurrent TCP **server** and **client** (`start_server`, `open_connection`)
- A tiny line-based **protocol** (a chat/echo server) and handling many clients at once
- **Async subprocesses**: `create_subprocess_exec`, streaming output, timeouts
- Async **HTTP** with `httpx.AsyncClient` (connection pooling, limits, retries)
- Async **file** I/O (`aiofiles` / `to_thread`)
- **anyio**: backend-independent structured concurrency

In [1]:
import asyncio, socket, sys, time, json

## 1. Networking in 60 seconds

| Concept | Meaning |
|---|---|
| **IP address** | which machine (`127.0.0.1` = this machine) |
| **Port** | which program on that machine (0–65535; < 1024 need privileges) |
| **TCP** | reliable, ordered **byte stream** (HTTP, SSH, databases) |
| **UDP** | individual **datagrams**, no delivery guarantee (DNS, games, video) |
| **Socket** | an endpoint for sending/receiving data |

TCP gives you a **stream of bytes**, not messages: you must define **framing** (e.g. "one JSON object per line").

## 2. Blocking sockets (the foundation)

In [2]:
import threading

def blocking_echo_server(sock):
    conn, addr = sock.accept()                  # blocks until a client connects
    with conn:
        data = conn.recv(1024)                  # blocks until data arrives
        conn.sendall(b"echo: " + data)

server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_sock.bind(("127.0.0.1", 0))              # port 0 → the OS picks a free port
server_sock.listen()
port = server_sock.getsockname()[1]
threading.Thread(target=blocking_echo_server, args=(server_sock,), daemon=True).start()

with socket.create_connection(("127.0.0.1", port), timeout=5) as client:
    client.sendall(b"hello sockets")
    print(client.recv(1024).decode())
server_sock.close()

echo: hello sockets


A blocking server needs **one thread per client**. asyncio handles thousands of connections in one thread.

## 3. An asyncio TCP server: a JSON-lines key-value store

In [3]:
STORE = {}

async def handle_client(reader: asyncio.StreamReader, writer: asyncio.StreamWriter):
    peer = writer.get_extra_info("peername")
    try:
        while line := await reader.readline():            # framing: one JSON command per line
            try:
                cmd = json.loads(line)
                match cmd:
                    case {"op": "set", "key": str(k), "value": v}:
                        STORE[k] = v
                        reply = {"ok": True}
                    case {"op": "get", "key": str(k)}:
                        reply = {"ok": k in STORE, "value": STORE.get(k)}
                    case {"op": "keys"}:
                        reply = {"ok": True, "keys": sorted(STORE)}
                    case _:
                        reply = {"ok": False, "error": "unknown command"}
            except json.JSONDecodeError:
                reply = {"ok": False, "error": "invalid JSON"}
            writer.write((json.dumps(reply) + "\n").encode())
            await writer.drain()                            # respect back-pressure
    finally:
        writer.close()
        await writer.wait_closed()

server = await asyncio.start_server(handle_client, "127.0.0.1", 0)
PORT = server.sockets[0].getsockname()[1]
print("KV server listening on port", PORT)

KV server listening on port 65325


## 4. An asyncio client

In [4]:
class KVClient:
    def __init__(self, host, port):
        self.host, self.port = host, port
    async def __aenter__(self):
        self.reader, self.writer = await asyncio.open_connection(self.host, self.port)
        return self
    async def __aexit__(self, *exc):
        self.writer.close()
        await self.writer.wait_closed()
    async def call(self, **cmd):
        self.writer.write((json.dumps(cmd) + "\n").encode())
        await self.writer.drain()
        return json.loads(await asyncio.wait_for(self.reader.readline(), timeout=5))

async with KVClient("127.0.0.1", PORT) as kv:
    print(await kv.call(op="set", key="lang", value="python"))
    print(await kv.call(op="get", key="lang"))
    print(await kv.call(op="dance"))

{'ok': True}
{'ok': True, 'value': 'python'}
{'ok': False, 'error': 'unknown command'}


### Many concurrent clients

In [5]:
async def client_session(i):
    async with KVClient("127.0.0.1", PORT) as kv:
        await kv.call(op="set", key=f"user{i}", value=i * i)
        return (await kv.call(op="get", key=f"user{i}"))["value"]

start = time.perf_counter()
values = await asyncio.gather(*(client_session(i) for i in range(200)))
print(f"200 concurrent clients served in {time.perf_counter() - start:.2f}s; sample: {values[:5]}")
async with KVClient("127.0.0.1", PORT) as kv:
    print("keys stored:", len((await kv.call(op="keys"))["keys"]))

200 concurrent clients served in 0.05s; sample: [0, 1, 4, 9, 16]
keys stored: 201


In [6]:
server.close()
await server.wait_closed()
print("server stopped")

server stopped


Key stream APIs: `reader.read(n)`, `readline()`, `readexactly(n)`, `readuntil(b"\r\n")`; `writer.write()` + **`await writer.drain()`**, `writer.close()` + `await writer.wait_closed()`. For TLS pass `ssl=ssl.create_default_context()`. Lower-level: `loop.create_server()` with `Protocol` classes; UDP: `loop.create_datagram_endpoint()`.

> 💡 Rather than inventing protocols, use existing ones: HTTP (FastAPI/aiohttp), WebSockets (`websockets` library), gRPC, or message brokers.

## 5. Async subprocesses

In [7]:
async def run(*cmd, timeout=10):
    proc = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE,
                                                stderr=asyncio.subprocess.PIPE)
    try:
        out, err = await asyncio.wait_for(proc.communicate(), timeout)
    except TimeoutError:
        proc.kill()
        await proc.wait()
        raise
    return proc.returncode, out.decode().strip()

# Run several commands CONCURRENTLY
results = await asyncio.gather(
    run(sys.executable, "-c", "import time; time.sleep(0.3); print('job A')"),
    run(sys.executable, "-c", "import time; time.sleep(0.3); print('job B')"),
    run("git", "--version"),
)
print(results)

[(0, 'job A'), (0, 'job B'), (0, 'git version 2.50.1 (Apple Git-155)')]


In [8]:
# Stream output line by line while the process runs
proc = await asyncio.create_subprocess_exec(
    sys.executable, "-u", "-c", "import time\nfor i in range(3):\n    print('progress', i * 50, '%')\n    time.sleep(0.1)",
    stdout=asyncio.subprocess.PIPE)
async for line in proc.stdout:
    print("live:", line.decode().strip())
print("exit code:", await proc.wait())

live: progress 0 %
live: progress 50 %


live: progress 100 %
exit code: 0


(`asyncio.create_subprocess_shell` exists too, with the same injection risks as `shell=True`, Module 11 · 06.)

## 6. Async HTTP with httpx

In [9]:
import httpx

async def fetch_json(client, url):
    r = await client.get(url)
    r.raise_for_status()
    return r.json()

limits = httpx.Limits(max_connections=10, max_keepalive_connections=5)
transport = httpx.AsyncHTTPTransport(retries=2)             # retry connection failures
async with httpx.AsyncClient(timeout=10, limits=limits, transport=transport,
                             headers={"User-Agent": "python-course"}) as client:     # one client = connection pool
    start = time.perf_counter()
    uuids = await asyncio.gather(*(fetch_json(client, "https://httpbin.org/uuid") for _ in range(5)))
    print(f"{len(uuids)} requests in {time.perf_counter() - start:.2f}s", uuids[0])

5 requests in 1.16s {'uuid': '7b3a89e3-e5de-42a5-b8f2-8bb1be62b087'}


- ✅ Reuse **one** `AsyncClient` per application (connection pooling); don't create one per request
- Set timeouts; limit concurrency (`Limits` and/or a `Semaphore`); handle `httpx.HTTPError`
- `client.stream("GET", url)` + `async for chunk in r.aiter_bytes()` for large downloads

## 7. Async file I/O

Operating systems don't offer truly async regular-file I/O everywhere, so async file libraries use threads under the hood:

In [10]:
from pathlib import Path

def write_report(path, text):
    Path(path).write_text(text, encoding="utf-8")

await asyncio.gather(*(asyncio.to_thread(write_report, f"report_{i}.txt", f"report {i}") for i in range(3)))
print(sorted(p.name for p in Path(".").glob("report_*.txt")))

import aiofiles                                    # pip install aiofiles
async with aiofiles.open("report_0.txt", encoding="utf-8") as f:
    print(await f.read())

['report_0.txt', 'report_1.txt', 'report_2.txt']
report 0


## 8. anyio: structured concurrency on any backend

`anyio` (used by httpx, Starlette/FastAPI) offers a trio-style API that runs on **asyncio or trio**:

In [11]:
import anyio

async def worker(name, delay, results):
    await anyio.sleep(delay)
    results.append(name)

results = []
async with anyio.create_task_group() as tg:          # like asyncio.TaskGroup
    for name, delay in [("a", 0.2), ("b", 0.1), ("c", 0.15)]:
        tg.start_soon(worker, name, delay, results)
print(results)

with anyio.move_on_after(0.1) as scope:              # cancel scope with a timeout
    await anyio.sleep(1)
print("cancelled by timeout:", scope.cancelled_caught)

['b', 'c', 'a']
cancelled by timeout: True


## 📝 Summary
- TCP is a byte stream: define framing (e.g. JSON lines). Blocking sockets need a thread per client; asyncio streams scale to thousands.
- `asyncio.start_server` / `open_connection`, `readline()`, `write()` + `drain()`, close properly.
- `create_subprocess_exec` runs and streams external programs concurrently.
- `httpx.AsyncClient` with pooling, limits, timeouts and retries for HTTP; `to_thread`/aiofiles for files.
- anyio provides structured concurrency independent of the event-loop library.

## 🏋️ Practice
1. Extend the KV server with `del` and `incr` commands and a per-client command counter.
2. Write a port scanner that checks ports 8000–8100 on localhost concurrently with `open_connection` and a timeout.
3. Download 10 URLs concurrently with httpx, saving each to a file with `aiofiles`, at most 3 at a time.
4. Run `python -m pytest` in several project folders concurrently with `create_subprocess_exec` and summarise the exit codes.